# Vibration-Based Event Detection + MASS Classification

**Detection**: Rolling 5s x-axis stddev > 5 = water is flowing (same logic as the frontend).
**Classification**: Once events are found, use `stumpy.mass` against calibration labels to identify fixture type.

In [10]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "requests", "numpy", "pandas", "scipy", "matplotlib", "stumpy"])


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


0

In [11]:
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
import matplotlib.pyplot as plt
import stumpy

HASURA_URL = "https://hasura.pipestuesday.org/v1/graphql"
HASURA_ADMIN_SECRET = "PIPE_SUPERMMMSECRET_PIPE"
HEADERS = {
    "Content-Type": "application/json",
    "x-hasura-admin-secret": HASURA_ADMIN_SECRET,
}

def gql_query(query, variables=None):
    payload = {"query": query}
    if variables:
        payload["variables"] = variables
    resp = requests.post(HASURA_URL, json=payload, headers=HEADERS)
    resp.raise_for_status()
    body = resp.json()
    if "errors" in body:
        raise RuntimeError(f"GraphQL errors: {body['errors']}")
    return body["data"]

print("Ready")

Ready


## 1. Fetch Last 2 Days of Data

In [12]:
SENSOR_ID = 6

# Last 2 days
from zoneinfo import ZoneInfo
eastern = ZoneInfo("America/New_York")
now_utc = datetime.now(timezone.utc)
start_utc = now_utc - timedelta(days=2)

# Fetch in chunks (Hasura limit 100k rows) — ~2 days at 10Hz = ~1.7M rows, need to paginate
all_reports = []
chunk_start = start_utc
CHUNK_HOURS = 6  # fetch 6h at a time

while chunk_start < now_utc:
    chunk_end = min(chunk_start + timedelta(hours=CHUNK_HOURS), now_utc)
    reports = gql_query("""
    query GetMag($sensor_id: bigint!, $since: timestamptz!, $until: timestamptz!) {
      mag_report(
        where: { sensor_id: {_eq: $sensor_id}, created_at: {_gte: $since, _lte: $until} }
        order_by: {created_at: asc}
        limit: 100000
      ) { created_at, x_axis_reading }
    }
    """, variables={"sensor_id": str(SENSOR_ID),
                    "since": chunk_start.isoformat(),
                    "until": chunk_end.isoformat()})["mag_report"]
    all_reports.extend(reports)
    print(f"  {chunk_start.strftime('%Y-%m-%d %H:%M')} -> {chunk_end.strftime('%H:%M')}: {len(reports)} rows")
    chunk_start = chunk_end

print(f"\nTotal: {len(all_reports)} readings")

df = pd.DataFrame(all_reports)
df["created_at"] = pd.to_datetime(df["created_at"], format="ISO8601")
df["x"] = pd.to_numeric(df["x_axis_reading"], errors="coerce")
df = df.dropna(subset=["x"]).reset_index(drop=True)

# Remove outliers (>3σ from median)
med = df["x"].median()
std = df["x"].std()
outlier_mask = (df["x"] - med).abs() > 3 * std
n_outliers = outlier_mask.sum()
df = df[~outlier_mask].reset_index(drop=True)
print(f"Removed {n_outliers} outlier(s)")

ts = df["created_at"].values
x = df["x"].values.astype(np.float64)
print(f"{len(x)} readings: {df['created_at'].min()} -> {df['created_at'].max()}")

  2026-03-31 14:59 -> 20:59: 100000 rows
  2026-03-31 20:59 -> 02:59: 100000 rows
  2026-04-01 02:59 -> 08:59: 100000 rows
  2026-04-01 08:59 -> 14:59: 100000 rows
  2026-04-01 14:59 -> 20:59: 100000 rows
  2026-04-01 20:59 -> 02:59: 100000 rows
  2026-04-02 02:59 -> 08:59: 100000 rows
  2026-04-02 08:59 -> 14:59: 100000 rows

Total: 800000 readings
Removed 2322 outlier(s)
797678 readings: 2026-03-31 14:59:53.628000+00:00 -> 2026-04-02 11:47:47.959000+00:00


## 2. Fetch Calibration Labels (Query Patterns)

Load labeled events the same way as the DTW notebook, then extract the x-axis signal for each as a query pattern.

In [13]:
# Fetch all calibration labels joined with their fixture (same as DTW notebook)
data = gql_query("""
query {
  calibration_label(order_by: {created_at: asc}) {
    id
    start_time
    end_time
    fixture {
      id
      name
      type
      sensor_id
    }
  }
}
""")

labels_raw = data["calibration_label"]
print(f"Total calibration labels: {len(labels_raw)}")

rows = []
for lab in labels_raw:
    fixture = lab.get("fixture") or {}
    rows.append({
        "label_id": lab["id"],
        "start_time": lab["start_time"],
        "end_time": lab["end_time"],
        "fixture_id": fixture.get("id"),
        "fixture_name": fixture.get("name"),
        "fixture_type": fixture.get("type"),
        "sensor_id": fixture.get("sensor_id"),
    })

df_labels = pd.DataFrame(rows)
df_labels["start_time"] = pd.to_datetime(df_labels["start_time"], format="ISO8601")
df_labels["end_time"] = pd.to_datetime(df_labels["end_time"], format="ISO8601")
df_labels["duration_s"] = (df_labels["end_time"] - df_labels["start_time"]).dt.total_seconds()

# Filter to our sensor
df_sensor = df_labels[df_labels["sensor_id"] == SENSOR_ID].dropna(subset=["fixture_type"]).reset_index(drop=True)
print(f"Labels for sensor {SENSOR_ID}: {len(df_sensor)}")
print(df_sensor["fixture_type"].value_counts().to_string())

Total calibration labels: 8
Labels for sensor 6: 8
fixture_type
sink      4
toilet    4


In [14]:
# Fetch mag data for each calibration label (same pattern as DTW notebook)
segments = []
for _, row in df_sensor.iterrows():
    sid = int(row["sensor_id"])
    t0 = row["start_time"].isoformat()
    t1 = row["end_time"].isoformat()

    mag_data = gql_query("""
    query GetMag($sensor_id: bigint!, $since: timestamptz!, $until: timestamptz!) {
      mag_report(
        where: { sensor_id: {_eq: $sensor_id}, created_at: {_gte: $since, _lte: $until} }
        order_by: {created_at: asc}
        limit: 100000
      ) { created_at, x_axis_reading }
    }
    """, variables={"sensor_id": str(sid), "since": t0, "until": t1})["mag_report"]

    if len(mag_data) == 0:
        print(f"  WARNING: label {row['label_id']} ({row['fixture_type']}) — 0 readings")
        continue

    df_mag = pd.DataFrame(mag_data)
    df_mag["x"] = pd.to_numeric(df_mag["x_axis_reading"], errors="coerce")
    sig = df_mag["x"].dropna().values.astype(np.float64)

    # Remove outliers from the query too
    med_s = np.median(sig)
    std_s = np.std(sig)
    sig = sig[np.abs(sig - med_s) <= 3 * std_s]

    if len(sig) < 10:
        continue

    segments.append({
        "label_id": row["label_id"],
        "fixture_type": row["fixture_type"],
        "fixture_name": row["fixture_name"],
        "signal": sig,
        "n_samples": len(sig),
        "ptp": np.ptp(sig),  # peak-to-peak range
    })

print(f"\nLoaded {len(segments)} query patterns:")
for s in segments:
    print(f"  {s['fixture_name']} ({s['fixture_type']}): {s['n_samples']} samples, range={s['ptp']:.1f}")

# Learn minimum event range from calibration data
cal_ranges = np.array([s["ptp"] for s in segments])
min_event_range = np.percentile(cal_ranges, 25) * 0.5
median_query_len = int(np.median([s["n_samples"] for s in segments]))

print(f"\nCalibration ranges: min={cal_ranges.min():.1f}, p25={np.percentile(cal_ranges, 25):.1f}, median={np.median(cal_ranges):.1f}")
print(f"Learned min_event_range: {min_event_range:.1f} (half of p25)")
print(f"Median query length: {median_query_len} samples")


Loaded 8 query patterns:
  men sink (sink): 183 samples, range=44.0
  women sink (sink): 154 samples, range=43.0
  men sink (sink): 150 samples, range=41.0
  women sink (sink): 165 samples, range=42.0
  men right toilet (toilet): 246 samples, range=45.0
  women left toilet (toilet): 244 samples, range=41.0
  men left toilet (toilet): 244 samples, range=39.0
  women right toilet (toilet): 250 samples, range=48.0

Calibration ranges: min=39.0, p25=41.0, median=42.5
Learned min_event_range: 20.5 (half of p25)
Median query length: 213 samples


## 3. Detect Events — High Jitter Regions

Rolling 3s stddev with quiet-baseline adaptive threshold (same logic as the backend detector).

In [15]:
from scipy.signal import savgol_filter, find_peaks

# Detection params (matched to backend detector/app/config.py)
ROLLING_STD_WINDOW_S = 3
JITTER_MULTIPLIER = 2.0
MIN_EVENT_GAP_S = 5
MIN_EVENT_LEN_S = 2
PAD_S = 4
LITRES_PER_CYCLE = 0.4

total_seconds = (pd.Timestamp(ts[-1]) - pd.Timestamp(ts[0])).total_seconds()
sps = len(x) / total_seconds
win = max(3, int(ROLLING_STD_WINDOW_S * sps))

# Rolling stddev
rolling_std = pd.Series(x).rolling(win, center=True, min_periods=3).std().fillna(0).values

# Quiet-baseline adaptive threshold (same as backend detection.py)
positive = rolling_std[rolling_std > 0]
if len(positive) > 0:
    sorted_std = np.sort(positive)
    quiet = sorted_std[:max(1, len(sorted_std) * 3 // 10)]  # bottom 30%
    quiet_mean = float(np.mean(quiet))
    quiet_std = float(np.std(quiet))
    threshold = quiet_mean + JITTER_MULTIPLIER * max(quiet_std, quiet_mean * 0.5)
else:
    threshold = 1.0
print(f"Rolling std: quiet_mean={quiet_mean:.4f}, quiet_std={quiet_std:.4f}, threshold={threshold:.4f}")

# Event = jitter above threshold
is_event = rolling_std > threshold

# Extract, merge, filter, pad
min_gap = int(MIN_EVENT_GAP_S * sps)
min_len = int(MIN_EVENT_LEN_S * sps)
pad = int(PAD_S * sps)

raw_runs = []
in_ev = False
for i in range(len(is_event)):
    if is_event[i] and not in_ev:
        ev_start = i; in_ev = True
    elif not is_event[i] and in_ev:
        raw_runs.append((ev_start, i - 1)); in_ev = False
if in_ev:
    raw_runs.append((ev_start, len(is_event) - 1))

merged = []
for s, e in raw_runs:
    if merged and (s - merged[-1][1]) <= min_gap:
        merged[-1] = (merged[-1][0], e)
    else:
        merged.append((s, e))

merged = [(s, e) for s, e in merged if (e - s) >= min_len]
merged = [(max(0, s - pad), min(len(x) - 1, e + pad)) for s, e in merged]

# Prevent padded events from overlapping
for i in range(1, len(merged)):
    prev_s, prev_e = merged[i - 1]
    curr_s, curr_e = merged[i]
    if curr_s <= prev_e:
        mid = (prev_e + curr_s) // 2
        merged[i - 1] = (prev_s, mid)
        merged[i] = (mid + 1, curr_e)

# Flow stats per event
event_stats = []
peak_indices_global = []

for s, e in merged:
    seg = x[s:e+1]
    if len(seg) < 5:
        event_stats.append({"n_peaks": 0, "n_cycles": 0, "volume_L": 0,
                            "duration_s": 0, "max_flow_lph": 0, "avg_flow_lph": 0})
        continue
    
    sg_win = min(7, len(seg) // 2 * 2 + 1)
    sm = savgol_filter(seg, window_length=max(3, sg_win), polyorder=min(3, max(3, sg_win) - 1)) if sg_win >= 3 else seg
    
    data_range = np.ptp(sm)
    pks = np.array([])
    if data_range > 0:
        pks, _ = find_peaks(sm, prominence=0.3 * data_range, distance=3)
    
    for pi in pks:
        peak_indices_global.append(s + pi)
    
    n_cycles = max(0, len(pks) - 1)
    volume_L = n_cycles * LITRES_PER_CYCLE
    t0 = pd.Timestamp(ts[s])
    t1 = pd.Timestamp(ts[min(e, len(ts)-1)])
    dur_s = (t1 - t0).total_seconds()
    
    if len(pks) >= 2:
        peak_ts_ms = np.array([(pd.Timestamp(ts[s + pi]) - t0).total_seconds() * 1000 for pi in pks])
        intervals = np.diff(peak_ts_ms)
        intervals = intervals[intervals > 0]
        max_flow_lph = LITRES_PER_CYCLE / (intervals.min() / 3_600_000) if len(intervals) > 0 else 0
        avg_flow_lph = LITRES_PER_CYCLE / (np.mean(intervals) / 3_600_000) if len(intervals) > 0 else 0
    else:
        max_flow_lph = avg_flow_lph = 0
    
    event_stats.append({"n_peaks": len(pks), "n_cycles": n_cycles, "volume_L": volume_L,
                        "duration_s": dur_s, "max_flow_lph": max_flow_lph, "avg_flow_lph": avg_flow_lph})

print(f"\nDetected {len(merged)} event(s):")
for i, ((s, e), st) in enumerate(zip(merged, event_stats)):
    t0 = pd.Timestamp(ts[s]); t1 = pd.Timestamp(ts[min(e, len(ts)-1)])
    print(f"  Event {i+1}: {t0.strftime('%H:%M:%S')} -> {t1.strftime('%H:%M:%S')} "
          f"({st['duration_s']:.1f}s, {st['n_cycles']} cycles, {st['volume_L']:.1f}L, "
          f"avg={st['avg_flow_lph']:.0f} L/h, max={st['max_flow_lph']:.0f} L/h)")


Rolling std: quiet_mean=1.2149, quiet_std=0.1646, threshold=2.4297

Detected 1693 event(s):
  Event 1: 14:59:53 -> 14:59:57 (3.9s, 0 cycles, 0.0L, avg=0 L/h, max=0 L/h)
  Event 2: 15:00:25 -> 15:00:31 (6.0s, 0 cycles, 0.0L, avg=0 L/h, max=0 L/h)
  Event 3: 15:01:56 -> 15:03:00 (64.9s, 42 cycles, 16.8L, avg=1052 L/h, max=2057 L/h)
  Event 4: 15:04:48 -> 15:04:56 (7.6s, 1 cycles, 0.4L, avg=389 L/h, max=389 L/h)
  Event 5: 15:05:24 -> 15:06:21 (57.0s, 13 cycles, 5.2L, avg=410 L/h, max=720 L/h)
  Event 6: 15:06:45 -> 15:07:00 (14.9s, 2 cycles, 0.8L, avg=406 L/h, max=533 L/h)
  Event 7: 15:10:19 -> 15:10:31 (11.5s, 0 cycles, 0.0L, avg=0 L/h, max=0 L/h)
  Event 8: 15:10:31 -> 15:10:36 (5.4s, 0 cycles, 0.0L, avg=0 L/h, max=0 L/h)
  Event 9: 15:16:21 -> 15:16:28 (7.1s, 1 cycles, 0.4L, avg=497 L/h, max=497 L/h)
  Event 10: 15:17:09 -> 15:17:14 (4.4s, 0 cycles, 0.0L, avg=0 L/h, max=0 L/h)
  Event 11: 15:17:14 -> 15:17:19 (5.3s, 0 cycles, 0.0L, avg=0 L/h, max=0 L/h)
  Event 12: 15:17:21 -> 15:17:

## 4. Classify Events with MASS

For each detected event, use `stumpy.mass` to find which calibration pattern it best matches.

In [16]:
# Classify each detected event by finding the best-matching calibration pattern
event_labels = []

for s, e in merged:
    event_signal = x[s:e+1]
    
    if len(event_signal) < 10:
        event_labels.append({"type": "unknown", "name": "?", "dist": np.inf})
        continue
    
    best_dist = np.inf
    best_seg = None
    
    for seg in segments:
        q = seg["signal"]
        # Query must be shorter than event (or we compare event against query)
        if len(q) <= len(event_signal):
            # How well does the calibration pattern match inside this event?
            dp = stumpy.mass(q, event_signal)
            d = dp.min()
        elif len(event_signal) <= len(q):
            # Event is shorter than query — match event inside query
            dp = stumpy.mass(event_signal, q)
            d = dp.min()
        else:
            continue
        
        if d < best_dist:
            best_dist = d
            best_seg = seg
    
    if best_seg is not None:
        event_labels.append({
            "type": best_seg["fixture_type"],
            "name": best_seg["fixture_name"],
            "dist": best_dist,
        })
    else:
        event_labels.append({"type": "unknown", "name": "?", "dist": np.inf})

print(f"Detected {len(merged)} event(s):")
for i, ((s, e), lbl) in enumerate(zip(merged, event_labels)):
    t0 = pd.Timestamp(ts[s])
    t1 = pd.Timestamp(ts[min(e, len(ts)-1)])
    dur = (t1 - t0).total_seconds()
    print(f"  Event {i+1}: {t0.strftime('%H:%M:%S')} -> {t1.strftime('%H:%M:%S')} ({dur:.1f}s) "
          f"— {lbl['type']} (match: {lbl['name']}, dist={lbl['dist']:.2f})")

Detected 1693 event(s):
  Event 1: 14:59:53 -> 14:59:57 (3.9s) — sink (match: men sink, dist=5.70)
  Event 2: 15:00:25 -> 15:00:31 (6.0s) — sink (match: women sink, dist=5.46)
  Event 3: 15:01:56 -> 15:03:00 (64.9s) — toilet (match: women left toilet, dist=11.81)
  Event 4: 15:04:48 -> 15:04:56 (7.6s) — sink (match: women sink, dist=8.67)
  Event 5: 15:05:24 -> 15:06:21 (57.0s) — sink (match: women sink, dist=12.56)
  Event 6: 15:06:45 -> 15:07:00 (14.9s) — toilet (match: men right toilet, dist=13.05)
  Event 7: 15:10:19 -> 15:10:31 (11.5s) — sink (match: men sink, dist=12.55)
  Event 8: 15:10:31 -> 15:10:36 (5.4s) — sink (match: men sink, dist=7.08)
  Event 9: 15:16:21 -> 15:16:28 (7.1s) — sink (match: men sink, dist=8.29)
  Event 10: 15:17:09 -> 15:17:14 (4.4s) — sink (match: men sink, dist=6.64)
  Event 11: 15:17:14 -> 15:17:19 (5.3s) — sink (match: men sink, dist=6.00)
  Event 12: 15:17:21 -> 15:17:33 (12.5s) — sink (match: women sink, dist=12.10)
  Event 13: 15:17:44 -> 15:17:53 (

## 5. Results Summary

In [17]:
# Summary table of all detected events
print(f"{'#':<4} {'Start':<22} {'End':<22} {'Dur(s)':>7} {'Type':<10} {'Cycles':>7} {'Volume':>8} {'Avg L/h':>8} {'Max L/h':>8} {'MASS dist':>10}")
print("-" * 115)
total_volume = 0
total_cycles = 0
for i, ((s, e), lbl, st) in enumerate(zip(merged, event_labels, event_stats)):
    t0 = pd.Timestamp(ts[s]).strftime('%Y-%m-%d %H:%M:%S')
    t1 = pd.Timestamp(ts[min(e, len(ts)-1)]).strftime('%H:%M:%S')
    total_volume += st['volume_L']
    total_cycles += st['n_cycles']
    print(f"{i+1:<4} {t0:<22} {t1:<22} {st['duration_s']:>7.1f} {lbl['type']:<10} "
          f"{st['n_cycles']:>7} {st['volume_L']:>7.1f}L {st['avg_flow_lph']:>7.0f} {st['max_flow_lph']:>7.0f} {lbl['dist']:>10.2f}")

print("-" * 115)
print(f"{'TOTAL':<4} {'':<22} {'':<22} {'':<7} {'':<10} {total_cycles:>7} {total_volume:>7.1f}L")
print(f"\n{len(merged)} events detected over {total_seconds/3600:.1f} hours")

#    Start                  End                     Dur(s) Type        Cycles   Volume  Avg L/h  Max L/h  MASS dist
-------------------------------------------------------------------------------------------------------------------
1    2026-03-31 14:59:53    14:59:57                   3.9 sink             0     0.0L       0       0       5.70
2    2026-03-31 15:00:25    15:00:31                   6.0 sink             0     0.0L       0       0       5.46
3    2026-03-31 15:01:56    15:03:00                  64.9 toilet          42    16.8L    1052    2057      11.81
4    2026-03-31 15:04:48    15:04:56                   7.6 sink             1     0.4L     389     389       8.67
5    2026-03-31 15:05:24    15:06:21                  57.0 sink            13     5.2L     410     720      12.56
6    2026-03-31 15:06:45    15:07:00                  14.9 toilet           2     0.8L     406     533      13.05
7    2026-03-31 15:10:19    15:10:31                  11.5 sink             0     0.

## 6. Clear Old Events + Store New Ones in `ts2vec_data`

In [18]:
# Delete all existing events for this sensor, then batch insert fresh detections
print("Deleting old events...")
result = gql_query("""
mutation DeleteOld($sensor_id: bigint!) {
  delete_ts2vec_data(where: {sensor_id: {_eq: $sensor_id}}) {
    affected_rows
  }
}
""", variables={"sensor_id": str(SENSOR_ID)})
deleted = result["delete_ts2vec_data"]["affected_rows"]
print(f"Deleted {deleted} old events")

# Batch insert all new detections
objects = []
for s, e in merged:
    objects.append({
        "sensor_id": str(SENSOR_ID),
        "start_time": pd.Timestamp(ts[s]).isoformat(),
        "end_time": pd.Timestamp(ts[min(e, len(ts) - 1)]).isoformat(),
    })

result = gql_query("""
mutation InsertEvents($objects: [ts2vec_data_insert_input!]!) {
  insert_ts2vec_data(objects: $objects) {
    affected_rows
  }
}
""", variables={"objects": objects})
inserted = result["insert_ts2vec_data"]["affected_rows"]
print(f"Inserted {inserted} events")


Deleting old events...
Deleted 667 old events
Inserted 1693 events
